In [ ]:
# %%
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

model = joblib.load("xgb_mfcc.pkl")

Model loaded!


In [20]:
# %%
SPEECH_MIXED_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mfcc/aurora_mfcc_speech_mixed.npy"

X_speech_new = np.load(SPEECH_MIXED_PATH, mmap_mode="r")

print("Speech mixed:", X_speech_new.shape)

# %%
X_test_new = np.vstack([X_speech_new]).astype(np.float32)

y_test_new = np.ones(len(X_speech_new), dtype=np.int8)

print("New test shape:", X_test_new.shape)
print("Distribution:", np.bincount(y_test_new))

Speech mixed: (45891, 39)
New test shape: (45891, 39)
Distribution: [    0 45891]


In [21]:
# # %%
# SPEECH_MIXED_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mfcc/mini_mfcc_speech_mixed.npy"
# NON_MIXED_PATH    = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mfcc/mini_mfcc_non-speech_mixed.npy"

# X_speech_new = np.load(SPEECH_MIXED_PATH, mmap_mode="r")
# X_non_new    = np.load(NON_MIXED_PATH, mmap_mode="r")

# print("Speech mixed:", X_speech_new.shape)
# print("Non mixed   :", X_non_new.shape)

# # %%
# X_test_new = np.vstack([X_non_new, X_speech_new]).astype(np.float32)

# y_test_new = np.hstack([
#     np.zeros(len(X_non_new), dtype=np.int8),
#     np.ones(len(X_speech_new), dtype=np.int8)
# ])

# print("New test shape:", X_test_new.shape)
# print("Distribution:", np.bincount(y_test_new))

In [22]:
# %%
batch_size = 200_000

y_pred_all = []
y_prob_all = []

for i in range(0, len(X_test_new), batch_size):
    Xb = X_test_new[i:i+batch_size]
    y_pred_all.append(model.predict(Xb))
    y_prob_all.append(model.predict_proba(Xb)[:, 1])

y_pred = np.concatenate(y_pred_all)
y_prob = np.concatenate(y_prob_all)

print("Prediction done!")

Prediction done!


In [23]:
# %%
print("=== MIXED TEST RESULT ===")
print("Accuracy :", accuracy_score(y_test_new, y_pred))
print("Precision:", precision_score(y_test_new, y_pred))
print("Recall   :", recall_score(y_test_new, y_pred))
print("F1-score :", f1_score(y_test_new, y_pred))
print("AUC      :", roc_auc_score(y_test_new, y_prob))

=== MIXED TEST RESULT ===
Accuracy : 0.9907171340785775
Precision: 1.0
Recall   : 0.9907171340785775
F1-score : 0.9953369236831735
AUC      : nan


/home/quochuy/miniconda3/envs/audio_ml/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [24]:
y_pred = (y_prob > 0.3).astype(int)
print("Recall new:", recall_score(y_test_new, y_pred))

Recall new: 0.9963609422326818
